# Flight Delay Prediction — Data Preparation & EDA

**Purpose:** One-time notebook. Loads the full feature-encoded parquets, runs EDA checks,  
and produces a reproducible stratified 10 % training sample saved back to Drive.  
Run this once (or whenever upstream features change). All modelling notebooks read  
the sample parquet — they never touch the 31 M-row full training set.

**Outputs (written to Drive):**
- `train_sample.parquet` — stratified 10 % of train, seed=42
- `val.parquet` — full validation set (2024); copied as-is for fast local access in model notebooks

**Test set is intentionally excluded** until final model evaluation.

---
## 1. Environment Setup

In [1]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq
!pip install pyspark --quiet

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 118252 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.30+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.30+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/rmid to provide /usr/bin/rmid

In [2]:
import os, subprocess
result = subprocess.run(
    "java -XshowSettings:property -version 2>&1 | grep 'java.home'",
    shell=True, capture_output=True, text=True
)
os.environ['JAVA_HOME'] = result.stdout.strip().split('=')[-1].strip()
print('JAVA_HOME:', os.environ['JAVA_HOME'])

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('FlightDelay_DataPrep')
    .master('local[*]')
    .config('spark.driver.memory', '40g')           #
    .config('spark.sql.shuffle.partitions', '64')   # larger data; more partitions
    .getOrCreate()
)
spark.conf.set('spark.sql.parquet.int96RebaseModeInRead', 'CORRECTED')
spark.conf.set('spark.sql.legacy.parquet.nanosAsLong', 'true')
spark.conf.set('spark.sql.parquet.mergeSchema', 'false')
spark.conf.set('spark.hadoop.parquet.enable.summary-metadata', 'false')
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

Spark version: 4.0.2


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
### Uncomment and use in case of crash with Drive connection
#from google.colab import drive
#drive.flush_and_unmount()
#drive.mount('/content/drive', force_remount=True)

---
## 2. Load Data

Copies train, val, and test from Drive to local NVMe SSD first.  
Drive I/O is slow (~50 MB/s); all subsequent Spark reads run off fast local storage.  
**Test is loaded here only for the null audit and scaling check.** It is not sampled  
or written to Drive — model notebooks will load it directly when needed.

Update `DATA_PATH` and `OUT_PATH` if your Drive folder structure differs.

In [6]:
from pyspark.sql import functions as F
import shutil, os

DATA_PATH = '/content/drive/MyDrive/OMDS Capstone/Data/flights_all_features_encoded'
OUT_PATH  = '/content/drive/MyDrive/OMDS Capstone/Data/flights_sample'
LOCAL     = '/content/local_data'
os.makedirs(LOCAL, exist_ok=True)

# Copy all three splits to fast local SSD
for name in ['train', 'val', 'test']:
    shutil.copytree(
        f'{DATA_PATH}/{name}.parquet',
        f'{LOCAL}/{name}.parquet',
        dirs_exist_ok=True
    )
    print(f'Copied {name} to local SSD')

train_df = spark.read.parquet(f'{LOCAL}/train.parquet')
val_df   = spark.read.parquet(f'{LOCAL}/val.parquet')
test_df  = spark.read.parquet(f'{LOCAL}/test.parquet')

print(f'\nRow counts (lazy — Spark will scan on first action):')
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'  {name:<6}: {df.count():>12,}')

Copied train to local SSD
Copied val to local SSD
Copied test to local SSD

Row counts (lazy — Spark will scan on first action):
  train :   31,149,502
  val   :    6,743,403
  test  :    6,965,246


---
## 3. Schema & Column Validation

Confirms the upstream feature pipeline produced all expected columns  
before any downstream work is attempted.

In [7]:
print(f"{'Column':<40} {'Type'}")
print('-' * 55)
for field in train_df.schema.fields:
    print(f'{field.name:<40} {field.dataType.simpleString()}')

Column                                   Type
-------------------------------------------------------
Year                                     bigint
Quarter                                  bigint
Month                                    bigint
DayofMonth                               bigint
DayOfWeek                                bigint
FlightDate                               bigint
Reporting_Airline                        string
Flight_Number_Reporting_Airline          string
Origin                                   string
Dest                                     string
CRSDepTime                               bigint
DepTimeBlk                               string
CRSArrTime                               bigint
ArrDel15                                 bigint
CRSElapsedTime                           double
Distance                                 double
DistanceGroup                            bigint
date                                     string
dep_hour                          

In [8]:
required = [
    'ArrDel15',
    'Month', 'DayOfWeek', 'dep_hour', 'Distance', 'CRSElapsedTime',
    'carrier_delay_rate_30d', 'carrier_delay_rate_90d',
    'origin_departures_3h',
    'origin_delay_rate', 'dest_delay_rate',
    'Year',  # required for stratification
]
missing = [c for c in required if c not in train_df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
print('All required columns present.')

All required columns present.


---
## 4. Feature Definition

Canonical feature list — kept in sync with the modelling notebooks.  
Every column here is provably observable at T–2 (two hours before departure).

| Group | Features |
|---|---|
| Schedule / calendar | `Month`, `DayOfWeek`, `dep_hour`, `Distance`, `CRSElapsedTime`, `is_weekend`, `is_holiday` |
| Carrier rolling | `carrier_delay_rate_30d`, `carrier_delay_rate_90d` |
| Airport congestion | `origin_departures_3h` |
| Target encoded | `origin_delay_rate`, `dest_delay_rate` |
| Weather flags (origin) | `origin_is_rain`, `origin_is_snow`, `origin_is_fog`, `origin_low_visibility`, `origin_high_wind`, `origin_severe_weather` |
| Weather flags (dest) | `dest_is_rain`, `dest_is_snow`, `dest_is_fog`, `dest_low_visibility`, `dest_high_wind`, `dest_severe_weather` |
| Weather continuous (origin) | `origin_visibility`, `origin_wind_kts`, `origin_gust_kts`, `origin_precip_in` |
| Weather continuous (dest) | `dest_visibility`, `dest_wind_kts`, `dest_gust_kts`, `dest_precip_in` |

**Excluded:** post-departure actuals, raw string identifiers, `Year`/`FlightDate`, thermal weather columns.

In [9]:
SCHEDULE_FEATURES = [
    'Month', 'DayOfWeek', 'dep_hour', 'Distance', 'CRSElapsedTime',
    'is_weekend', 'is_holiday',
]
ENGINEERED_FEATURES = [
    'carrier_delay_rate_30d', 'carrier_delay_rate_90d',
    'origin_departures_3h',
    'origin_delay_rate', 'dest_delay_rate',
]
WEATHER_FLAGS = [
    'origin_is_rain', 'origin_is_snow', 'origin_is_fog',
    'origin_low_visibility', 'origin_high_wind', 'origin_severe_weather',
    'dest_is_rain', 'dest_is_snow', 'dest_is_fog',
    'dest_low_visibility', 'dest_high_wind', 'dest_severe_weather',
]
WEATHER_CONTINUOUS = [
    'origin_visibility', 'origin_wind_kts', 'origin_gust_kts', 'origin_precip_in',
    'dest_visibility',   'dest_wind_kts',   'dest_gust_kts',   'dest_precip_in',
]
FEATURE_COLS = SCHEDULE_FEATURES + ENGINEERED_FEATURES + WEATHER_FLAGS + WEATHER_CONTINUOUS
TARGET_COL   = 'ArrDel15'

print(f'Total features : {len(FEATURE_COLS)}')
print(f'  Schedule     : {len(SCHEDULE_FEATURES)}')
print(f'  Engineered   : {len(ENGINEERED_FEATURES)}')
print(f'  Weather flags: {len(WEATHER_FLAGS)}')
print(f'  Weather cont : {len(WEATHER_CONTINUOUS)}')

Total features : 32
  Schedule     : 7
  Engineered   : 5
  Weather flags: 12
  Weather cont : 8


---
## 5. EDA Checks

Three checks run across all splits before any sampling:

1. **Null audit** — single aggregation per split (3 Spark actions total)  
2. **Class balance** — delayed vs not-delayed ratio across splits  
3. **Scaling verification** — confirms features are not pre-standardised,  
   so `StandardScaler` in the model pipeline will behave as intended

In [10]:
# ── 5a. Null audit ────────────────────────────────────────────────────────────
audit_cols = FEATURE_COLS + [TARGET_COL]

def null_counts(df):
    exprs = [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in audit_cols]
    return df.select(exprs).collect()[0].asDict()

train_nulls = null_counts(train_df)
val_nulls   = null_counts(val_df)
test_nulls  = null_counts(test_df)

print(f"{'Column':<30} {'train':>10} {'val':>10} {'test':>10}")
print('-' * 64)
for col in audit_cols:
    tr, va, te = train_nulls[col], val_nulls[col], test_nulls[col]
    flag = '  <-- WARNING' if any([tr, va, te]) else ''
    print(f'{col:<30} {tr:>10,} {va:>10,} {te:>10,}{flag}')

Column                              train        val       test
----------------------------------------------------------------
Month                                   0          0          0
DayOfWeek                               0          0          0
dep_hour                                0          0          0
Distance                                0          0          0
CRSElapsedTime                          0          0          0
is_weekend                              0          0          0
is_holiday                              0          0          0
carrier_delay_rate_30d                  0          0          0
carrier_delay_rate_90d                  0          0          0
origin_departures_3h                    0          0          0
origin_delay_rate                       0          0          0
dest_delay_rate                         0          0          0
origin_is_rain                          0          0          0
origin_is_snow                         

In [11]:
# ── 5b. Class balance ─────────────────────────────────────────────────────────
print(f"{'Split':<8} {'Not delayed':>14} {'Delayed':>12} {'Delay %':>10}")
print('-' * 48)
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    counts = {r[TARGET_COL]: r['count'] for r in df.groupBy(TARGET_COL).count().collect()}
    neg, pos = counts.get(0, 0), counts.get(1, 0)
    pct = 100 * pos / (neg + pos) if (neg + pos) > 0 else 0
    print(f'{name:<8} {neg:>14,} {pos:>12,} {pct:>9.2f}%')

Split       Not delayed      Delayed    Delay %
------------------------------------------------
train        25,589,033    5,560,469     17.85%
val           5,356,704    1,386,699     20.56%
test          5,515,280    1,449,966     20.82%


In [12]:
# ── 5c. Scaling verification ─────────────────────────────────────────────────
# Sample 5 000 rows and check value ranges across all features.
# If features were already standardised they would cluster near mean≈0, std≈1.
# Raw engineering values (Distance in miles, dep_hour 0-23, rates 0-1) will show
# clearly distinct ranges, confirming StandardScaler has not been pre-applied.
import pandas as pd

sample_pd = train_df.select(FEATURE_COLS).limit(5_000).toPandas()

stats = sample_pd.describe().T[['mean', 'std', 'min', 'max']].round(4)
stats['scaled?'] = (
    (stats['mean'].abs() < 0.1) & (stats['std'].between(0.9, 1.1))
).map({True: '⚠ possibly', False: 'no'})

print(f"{'Feature':<30} {'mean':>10} {'std':>10} {'min':>10} {'max':>10}  scaled?")
print('-' * 84)
for feat, row in stats.iterrows():
    print(f"{feat:<30} {row['mean']:>10.4f} {row['std']:>10.4f} "
          f"{row['min']:>10.4f} {row['max']:>10.4f}  {row['scaled?']}")

n_suspicious = (stats['scaled?'] == '⚠ possibly').sum()
if n_suspicious == 0:
    print('\n✓ No features appear pre-scaled — StandardScaler will be applied as intended.')
else:
    print(f'\n⚠ {n_suspicious} feature(s) look possibly pre-scaled — inspect before proceeding.')

Feature                              mean        std        min        max  scaled?
------------------------------------------------------------------------------------
Month                              1.0000     0.0000     1.0000     1.0000  no
DayOfWeek                          3.2416     1.5372     1.0000     6.0000  no
dep_hour                          11.7170     4.6105     0.0000    23.0000  no
Distance                         659.9374   474.3585    83.0000  4502.0000  no
CRSElapsedTime                   126.7776    51.2509    55.0000   520.0000  no
is_weekend                         0.0706     0.2562     0.0000     1.0000  no
is_holiday                         0.3690     0.4826     0.0000     1.0000  no
carrier_delay_rate_30d             0.1998     0.0651     0.1195     0.5105  no
carrier_delay_rate_90d             0.1998     0.0651     0.1195     0.5105  no
origin_departures_3h              30.1728    35.2467     0.0000   190.0000  no
origin_delay_rate                  0.2625

---
## 6. Create Stratified 20 % Training Sample

`sampleBy` draws from each `(Year, Month)` stratum independently at exactly 20 %.  
The resulting parquet (~6.2 M rows) is saved to Drive and reused by all model notebooks,  

Val is copied to the same output folder for convenience — model notebooks read both  
from `flights_sample/` and never need to access `flights_all_features_encoded/` directly.

In [13]:
# Tag each row with a (Year_Month) stratum key
train_tagged = train_df.withColumn(
    '_stratum',
    F.concat_ws('_', F.col('Year').cast('string'), F.col('Month').cast('string'))
)

# Collect all unique strata and assign 10% fraction to each
strata_keys = [r['_stratum'] for r in train_tagged.select('_stratum').distinct().collect()]
fractions   = {key: 0.20 for key in strata_keys}
print(f'Strata found: {len(strata_keys)}  (one per Year×Month combination)')

train_sub = (
    train_tagged
    .sampleBy('_stratum', fractions=fractions, seed=42)
    .drop('_stratum')
)

# Materialise to local SSD first (faster write), then copy to Drive
LOCAL_SAMPLE = f'{LOCAL}/train_sample.parquet'
train_sub.coalesce(10).write.mode('overwrite').parquet(LOCAL_SAMPLE)
train_sub = spark.read.parquet(LOCAL_SAMPLE)
print(f'Sample row count : {train_sub.count():>10,}')
print(f'Full train rows  : {train_df.count():>10,}')

Strata found: 60  (one per Year×Month combination)
Sample row count :  6,231,167
Full train rows  : 31,149,502


In [14]:
# Verify class balance is preserved in the sample
print('Class balance — full train vs sample:')
print(f"{'Split':<12} {'Not delayed':>14} {'Delayed':>12} {'Delay %':>10}")
print('-' * 52)
for name, df in [('full train', train_df), ('sample', train_sub)]:
    counts = {r[TARGET_COL]: r['count'] for r in df.groupBy(TARGET_COL).count().collect()}
    neg, pos = counts.get(0, 0), counts.get(1, 0)
    pct = 100 * pos / (neg + pos)
    print(f'{name:<12} {neg:>14,} {pos:>12,} {pct:>9.2f}%')

# Verify stratum coverage
print('\nStratum row counts in sample:')
(
    train_sub
    .withColumn('_stratum', F.concat_ws('_', F.col('Year').cast('string'), F.col('Month').cast('string')))
    .groupBy('_stratum').count()
    .orderBy('_stratum')
    .show(50, truncate=False)
)

Class balance — full train vs sample:
Split           Not delayed      Delayed    Delay %
----------------------------------------------------
full train       25,589,033    5,560,469     17.85%
sample            5,116,995    1,114,172     17.88%

Stratum row counts in sample:
+--------+------+
|_stratum|count |
+--------+------+
|2018_1  |110612|
|2018_10 |121829|
|2018_11 |115445|
|2018_12 |116975|
|2018_2  |102671|
|2018_3  |118739|
|2018_4  |117640|
|2018_5  |122033|
|2018_6  |122183|
|2018_7  |125881|
|2018_8  |124499|
|2018_9  |115257|
|2019_1  |112715|
|2019_10 |126042|
|2019_11 |119473|
|2019_12 |123391|
|2019_2  |103132|
|2019_3  |123157|
|2019_4  |119228|
|2019_5  |124385|
|2019_6  |124053|
|2019_7  |128424|
|2019_8  |129260|
|2019_9  |118851|
|2020_1  |120351|
|2020_10 |69931 |
|2020_11 |71924 |
|2020_12 |73507 |
|2020_2  |113613|
|2020_3  |107632|
|2020_4  |36553 |
|2020_5  |33802 |
|2020_6  |44350 |
|2020_7  |70056 |
|2020_8  |74212 |
|2020_9  |64521 |
|2021_1  |71565 |
|2

In [15]:
# Write sample and val to Drive
import shutil

os.makedirs(OUT_PATH, exist_ok=True)

# train_sample: copy from local SSD to Drive
DRIVE_SAMPLE = f'{OUT_PATH}/train_sample.parquet'
if os.path.exists(DRIVE_SAMPLE):
    shutil.rmtree(DRIVE_SAMPLE)
shutil.copytree(LOCAL_SAMPLE, DRIVE_SAMPLE)
print(f'Written: {DRIVE_SAMPLE}')

# val: copy full val set to the same output folder for model notebooks
DRIVE_VAL = f'{OUT_PATH}/val.parquet'
if os.path.exists(DRIVE_VAL):
    shutil.rmtree(DRIVE_VAL)
shutil.copytree(f'{LOCAL}/val.parquet', DRIVE_VAL)
print(f'Written: {DRIVE_VAL}')

print('\nData prep complete. Model notebooks can now read from:')
print(f'  {OUT_PATH}/')

Written: /content/drive/MyDrive/OMDS Capstone/Data/flights_sample/train_sample.parquet
Written: /content/drive/MyDrive/OMDS Capstone/Data/flights_sample/val.parquet

Data prep complete. Model notebooks can now read from:
  /content/drive/MyDrive/OMDS Capstone/Data/flights_sample/
